# Creating a Simple Agent with Tracing

In [12]:
import dotenv
import os

from openai import OpenAI

dotenv.load_dotenv()

if not os.environ.get("OPENAI_API_KEY"):
    print(
        """Error: OPENAI_API_KEY environment variable not set. Please copy the .env.template file as .env and fill it in.
    
    You can execute these commands in the terminal to get started:
    cp .env.template .env
    code .env
    """
    )

# Test OpenAI Access
print(
    OpenAI()
    .responses.create(
        model=os.environ["OPENAI_DEFAULT_MODEL"], input="Say: We are up and running!"
    )
    .output_text
)

We are up and running!


In [13]:
from agents import Agent, Runner, trace
from openai.types.responses import ResponseTextDeltaEvent

Create a simple Nutrition Assistant Agent

In [ ]:
nutrition_agent = Agent(
    name="Nutrition Assistant",
    instructions="""
    You are a helpful assistant giving out nutrition advice.
    You give concise answers.
    """,
)

Let's execute the Agent:

In [15]:
with trace("Simple Nutrition Agent"):
    result = await Runner.run(nutrition_agent, "How healthy are bananas?")

print(result)

RunResult:
- Last agent: Agent(name="Nutrition Assistant", ...)
- Final output (str):
    Bananas are nutritious and convenient. Key points:
    
    - Good: potassium (helps blood pressure), vitamin B6, vitamin C, fiber, and resistant starch in greener bananas. Low in fat.
    - Calories: about 90–110 per medium banana; sugars and carbs are moderate.
    - Benefits: supports heart health, digestion, and energy; portable snack.
    - Considerations: ripe bananas are higher in sugar; GI is moderate, so portion matters for some diabetics.
    - Tips: pair with protein/fat (e.g., peanut butter) for steadier energy; if you have kidney disease or need potassium limits, monitor intake.
    
    Bottom line: healthy, naturally sweet, fits many diets in moderation.
- 2 new item(s)
- 1 raw response(s)
- 0 input guardrail result(s)
- 0 output guardrail result(s)
(See `RunResult` for more details)


Streaming the answer to the screen, token by token

In [16]:
response_stream = Runner.run_streamed(nutrition_agent, "How healthy are bananas?")

async for event in response_stream.stream_events():
    if event.type == "raw_response_event" and isinstance(
        event.data, ResponseTextDeltaEvent
    ):
        print(event.data.delta, end="", flush=True)

Overall, yes—bananas are healthy as part of a balanced diet.

What you get per medium banana (about 118 g):
- Potassium: ~400–420 mg (helps blood pressure)
- Fiber: ~3 g
- Vitamin B6: ~0.5 mg
- Vitamin C: ~10% of daily value
- About 100–110 calories; sugars and carbs are moderate

Notes:
- Unripe bananas have more resistant starch; ripe ones are sweeter with more sugar.
- Potential downsides: can cause gas in some people; portion matters for those tracking carbs or sugar (diabetes, ketosis).
- Best eaten with variety: add to yogurt, oats, or smoothies, not as a sole staple.

Bottom line: a nutritious, convenient fruit that fits most healthy diets.

_Good Job!_